# ⛓️ Understanding Chains in LangChain (LCEL)

## Learning Objectives
In this notebook, you will learn:
1. **Basic LCEL Chains** - compose a prompt, model, and output parser with the `|` pipe operator
2. **Parallel Execution** - run multiple chains concurrently with `RunnableParallel`
3. **Passthrough Patterns** - forward original inputs alongside derived context with `RunnablePassthrough`
4. **Conditional Branching** - route inputs to different chains at runtime with `RunnableBranch`
5. **Chain Debugging** - inspect input/output schemas and trace intermediate execution steps

## Prerequisites
- Familiarity with LangChain basics (prompts, chat models, output parsers)
- `OPENAI_API_KEY` set in a project-root `.env` file
- `langchain`, `langchain-openai`, and `python-dotenv` installed

> Converted from `07_chains_v1.py` - part of **01 LangChain Foundations**.

---
## 🔧 Part 1: Environment Setup

Load environment variables and initialize the chat model shared by every demo in this notebook. `init_chat_model` provides a provider-agnostic way to construct a chat model from a simple model-name string, so the rest of the notebook doesn't need to care which provider backs `model`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API Keys and Initialize Chat Model
# ============================================================================
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)

load_dotenv()

model = init_chat_model(model="gpt-4o-mini", temperature=0)

print("✅ Environment variables loaded")
print("🤖 Model initialized: gpt-4o-mini (temperature=0)")

---
## ⛓️ Part 2: Core LCEL Chain Patterns

LCEL (LangChain Expression Language) lets you compose `Runnable` building blocks with the `|` pipe operator. The demos below build up from the simplest possible chain to parallel execution, passthrough context, conditional branching, and finally debugging techniques.

### 2.1 🧱 `demo_basic_chain`
The simplest LCEL chain: pipe a prompt into a model into an output parser. This is the foundational pattern every other chain in this notebook builds on.

### Key Concepts:
- **`|` (pipe)**: feeds the output of one `Runnable` into the input of the next
- **`StrOutputParser`**: extracts the plain string content from an `AIMessage`

In [ ]:
# ============================================================================
# BASIC CHAIN: Prompt -> Model -> Parser
# ============================================================================
def demo_basic_chain():
    prompt = ChatPromptTemplate.from_template(
        "Summarize the following text in one sentence: {text}"
    )

    parser = StrOutputParser()

    chain = prompt | model | parser

    result = chain.invoke(
        {
            "text": "LangChain is a framework for developing applications powered by language models."
        }
    )
    print(f"Summary: {result}  ")

In [ ]:
# ============================================================================
# RUN: Basic Chain Demo
# ============================================================================
demo_basic_chain()

### 2.2 🔀 `demo_parallel_chain`

Runs three independent prompts - summary, keyword extraction, and sentiment analysis - against the same input text at the same time via `RunnableParallel`, then collects all three outputs into a single result dictionary.

In [ ]:
# ============================================================================
# PARALLEL CHAIN: Run Multiple Chains Concurrently
# ============================================================================
def demo_parallel_chain():
    """Run multiple chains in parallel."""
    # define individual chains
    summarize_prompt = ChatPromptTemplate.from_template(
        "Summarize in two sentences: {text}"
    )
    keywords_prompt = ChatPromptTemplate.from_template(
        "Extract 5 keywords in the following text: {text}\nReturn as a comma-separated list."
    )
    sentiment_prompt = ChatPromptTemplate.from_template(
        "What is the sentiment of the following text? {text}"
    )

    parser = StrOutputParser()

    # Parallel execution
    analysis_chain = RunnableParallel(
        summary=summarize_prompt | model | parser,
        keywords=keywords_prompt | model | parser,
        sentiment=sentiment_prompt | model | parser,
    )

    text = """
    The new AI features are absolutely incredible! Users are loving the
    faster response times and improved accuracy. However, some have noted
    that the pricing could be more competitive. Overall, the product
    launch has been a massive success with record-breaking adoption rates.
    """

    results = analysis_chain.invoke({"text": text})
    print("Analysis Results:")
    print("Parallel Analysis Results:")
    print(f"  Summary: {results['summary']}")
    print(f"  Keywords: {results['keywords']}")
    print(f"  Sentiment: {results['sentiment']}")

In [ ]:
# ============================================================================
# RUN: Parallel Chain Demo
# ============================================================================
demo_parallel_chain()

### 2.3 🔁 `demo_passthrough_chain`

Simulates a retrieval step with a `RunnableLambda`, then uses `RunnableParallel` together with `RunnablePassthrough` to keep the original question available alongside the freshly retrieved context. This is a minimal preview of the pattern full RAG chains rely on.

In [ ]:
# ============================================================================
# PASSTHROUGH CHAIN: Preserve Original Input Alongside Derived Context
# ============================================================================
def demo_passthrough_chain():
    """A chain that demonstrates passthrough functionality."""
    prompt = ChatPromptTemplate.from_template(
        "Original question: {question}\n"
        "Context: {context}\n\n"
        "Answer the question based on the context."
    )

    # simulate a retrieve operation
    def fake_retriever(input_dict):
        return " LangChain was created by Harrison Chase in 2022."

    chain = (
        RunnableParallel(
            context=RunnableLambda(fake_retriever), question=RunnablePassthrough()
        )
        | RunnableLambda(
            lambda x: {"context": x["context"], "question": x["question"]["question"]}
        )
        | prompt
        | model
        | StrOutputParser()
    )

    result = chain.invoke({"question": "Who created LangChain?"})
    print(f"Answer: {result}")

In [ ]:
# ============================================================================
# RUN: Passthrough Chain Demo
# ============================================================================
demo_passthrough_chain()

### 2.4 🌿 `demo_chain_branching`

Uses `RunnableBranch` to route a question to a coding-focused chain or a general-purpose chain, based on the output of a lightweight classifier chain evaluated first.

In [ ]:
# ============================================================================
# BRANCHING CHAIN: Route to Different Chains Based on a Condition
# ============================================================================
def demo_chain_branching():
    """A chain that demonstrates branching functionality."""

    # Different prompts for different intents
    code_prompt = ChatPromptTemplate.from_template(
        "You are a coding expert. Help with: {input}"
    )
    general_prompt = ChatPromptTemplate.from_template(
        "You are a helpful assistant. Answer: {input}"
    )

    # Classifier
    classifier_prompt = ChatPromptTemplate.from_template(
        "Classify this as 'code' or 'general': {input}\nReturn only the classification."
    )
    classifer = classifier_prompt | model | StrOutputParser()

    # Branching chain  based on classification
    def is_code_question(input_dict):
        classification = classifer.invoke(input_dict)
        return "code" in classification.lower()

    branch = RunnableBranch(
        (is_code_question, code_prompt | model | StrOutputParser()),
        general_prompt | model | StrOutputParser(),  # default branch
    )

    # Test
    questions = [
        "How do I write a for loop in Python?",
        "What's the weather like today?",
    ]
    for q in questions:
        result = branch.invoke({"input": q})
        print(f"Q: {q}")
        print(f"A: {result[:100]}...\n")

In [ ]:
# ============================================================================
# RUN: Chain Branching Demo
# ============================================================================
demo_chain_branching()

### 2.5 🐞 `demo_debugging`

Shows three ways to introspect and debug an LCEL chain: reading its auto-generated input/output JSON schemas, attaching a run name via `with_config` for tracing tools like LangSmith, and inserting `RunnableLambda` logging steps between pipe stages to inspect intermediate values.

In [ ]:
# ============================================================================
# DEBUGGING: Inspect Schemas, Tag Runs, and Log Intermediate Steps
# ============================================================================
def demo_debugging():
    prompt = ChatPromptTemplate.from_template("Say hello to {name}")
    chain = prompt | model | StrOutputParser()

    # Method 1: Get configuration
    print("Chain input schema:", chain.input_schema.model_json_schema())
    print("Chain output schema:", chain.output_schema.model_json_schema())

    # Method 2: Use with_config for tacing
    result = chain.with_config(
        run_name="greeting_chain",
        # tags="demo,debugging",
    ).invoke({"name": "Alice"})
    print(f"Greeting: {result}")

    # Method 3: Inspect intermediate steps
    # Using RunnableLambda for logging
    def log_step(x, step_name=""):
        print(f"[{step_name}] {type(x).__name__}: {str(x)[:100]}")
        return x

    debug_chain = (
        prompt
        | RunnableLambda(lambda x: log_step(x, "after_prompt"))
        | model
        | RunnableLambda(lambda x: log_step(x, "after_model"))
        | StrOutputParser()
    )

    print("\nDebug chain execution:")
    result = debug_chain.invoke({"name": "Debug"})
    print(f"Greeting: {result}")

In [ ]:
# ============================================================================
# RUN: Debugging Demo
# ============================================================================
demo_debugging()

---
## 📝 Summary

In this notebook, we explored the core LCEL (LangChain Expression Language) chain-composition patterns.

### 1. Chain Composition Patterns
- **Basic chain**: `prompt | model | parser` - the fundamental LCEL building block
- **Parallel chains**: `RunnableParallel` runs independent chains concurrently and merges their results into one dict
- **Passthrough chains**: `RunnablePassthrough` preserves the original input alongside derived values (e.g. retrieved context) - the building block behind RAG-style chains
- **Branching chains**: `RunnableBranch` routes execution to different chains based on a runtime condition

### 2. Debugging Techniques
- Inspect `chain.input_schema` / `chain.output_schema` to understand a chain's expected input/output shape
- Use `chain.with_config(run_name=...)` to label runs for tracing tools like LangSmith
- Insert `RunnableLambda` logging steps between pipe stages to see intermediate values as they flow through the chain

### Functions Defined
- `demo_basic_chain()`
- `demo_parallel_chain()`
- `demo_passthrough_chain()`
- `demo_chain_branching()`
- `demo_debugging()`

### Next Steps
- See how these patterns combine into full retrieval-augmented generation (RAG) chains
- Explore `RunnableWithMessageHistory` for adding conversational memory on top of a chain
- Continue to the next notebook in **01 LangChain Foundations** for agent-based patterns built on these chains